# Evaluacion axial final v2 - test once

Evalua el held-out test una unica vez con confirmacion explicita. No entrena.

In [ ]:
import importlib.util
import os
import subprocess
import sys

PFI_INSTALL_NOTEBOOK_DEPS = os.getenv("PFI_INSTALL_NOTEBOOK_DEPS", "1") == "1"
REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "pydicom": "pydicom",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "nibabel": "nibabel",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and PFI_INSTALL_NOTEBOOK_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
elif missing:
    raise RuntimeError(f"Dependencias faltantes: {missing}")


In [ ]:
# Montaje idempotente de Google Drive para Colab. No hace nada en entorno local/VM.
from pathlib import Path as _Path

_PREFI_USE_GOOGLE_DRIVE = os.getenv("PFI_USE_GOOGLE_DRIVE", "1").strip().lower() in {"1", "true", "yes", "on"}
_IS_COLAB = importlib.util.find_spec("google.colab") is not None
_MYDRIVE = _Path("/content/drive/MyDrive")
if _IS_COLAB and _PREFI_USE_GOOGLE_DRIVE and not _MYDRIVE.exists():
    from google.colab import drive

    mount_root = _Path("/content/drive")
    mount_root.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mount_root), force_remount=False)
elif _IS_COLAB and _PREFI_USE_GOOGLE_DRIVE:
    print("Google Drive ya disponible; no se remonta.")
else:
    print("Google Drive mount omitido: entorno no Colab o PFI_USE_GOOGLE_DRIVE=False.")


In [ ]:
from __future__ import annotations

import csv
import dataclasses
import hashlib
import importlib
import json
import math
import os
import platform
import random
import re
import subprocess
import sys
import tempfile
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    import pydicom
except Exception:
    pydicom = None
try:
    import SimpleITK as sitk
except Exception:
    sitk = None
try:
    import nibabel as nib
except Exception:
    nib = None


In [ ]:
# Commit validado que contiene la implementacion runtime de AxialUNet2D
# y los helpers de carga de checkpoints usados por este entrenamiento.
AI_SERVICE_COMMIT_SHA = (
    "285159982832abb604a176b4302ac83a837ff1c9"
)

PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))

if not (PFI_REPO_ROOT / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git",
            str(PFI_REPO_ROOT),
        ],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(PFI_REPO_ROOT), "fetch", "origin"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(PFI_REPO_ROOT), "checkout", AI_SERVICE_COMMIT_SHA],
    check=True,
)

os.environ["PFI_REPO_ROOT"] = str(PFI_REPO_ROOT)

expected_architecture_file = (
    PFI_REPO_ROOT
    / "ai_service"
    / "pfi_ai_service"
    / "model_architectures.py"
)

if not expected_architecture_file.exists():
    raise FileNotFoundError(
        f"No se encontro el modulo esperado: {expected_architecture_file}"
    )

print("Repositorio AI Module:", PFI_REPO_ROOT)
print("AI_SERVICE_COMMIT_SHA:", AI_SERVICE_COMMIT_SHA)
print("model_architectures.py:", expected_architecture_file.name)


## Configuracion

In [ ]:
def env_bool(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    return default if raw is None else raw.strip().lower() in {"1", "true", "yes", "on"}


def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


PFI_CLOUD_MODE = env_bool("PFI_CLOUD_MODE", False)
PFI_USE_GOOGLE_DRIVE = env_bool("PFI_USE_GOOGLE_DRIVE", True)
PFI_SYNC_DRY_RUN = env_bool("PFI_SYNC_DRY_RUN", True)
DRIVE_ROOT = Path(os.getenv("PFI_DRIVE_ROOT", "/content/drive/MyDrive/PFI_MVP"))
EXTERNAL_ROOT = DRIVE_ROOT if PFI_USE_GOOGLE_DRIVE else Path(os.getenv("PFI_LOCAL_EXTERNAL_ROOT", "/tmp/pfi_mvp"))


In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    RUN_MODE: Literal["evaluate"] = os.getenv("RUN_MODE", "evaluate")
    SEED: int = int(os.getenv("PFI_SEED", "2026"))
    RUN_ID: str = os.getenv("PFI_RUN_ID", "axial-final-v2")
    REPO_ROOT: Path = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
    DATASET_ROOT: Path = Path(os.getenv("PFI_DATASET_ROOT", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    IMAGES_ROOT: Path = Path(os.getenv("AXIAL_IMAGES_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    MASKS_ROOT: Path = Path(os.getenv("AXIAL_MASKS_DIR", str(EXTERNAL_ROOT / "data" / "AXIAL_ALKAFRI")))
    SPLIT_MANIFEST_PATH: Path = Path(os.getenv("AXIAL_E9_CURATED_SPLIT_CSV", str(EXTERNAL_ROOT / "results" / "E9_alkafri_axial_t2_final_labels_baseline" / "E9_t2_final_labels_curated_split.csv")))
    OUTPUT_ROOT: Path = Path(os.getenv("PFI_OUTPUT_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_v2_training")))
    RESUME_ROOT: Path = Path(os.getenv("PFI_RESUME_ROOT", str(EXTERNAL_ROOT / "outputs" / "axial_final_v2_training" / "resume")))
    TARGET_SIZE: tuple[int, int] = (256, 256)
    NUM_CLASSES: int = 6
    BASE_CHANNELS: int = int(os.getenv("PFI_BASE_CHANNELS", "16"))
    BATCH_SIZE: int = int(os.getenv("PFI_BATCH_SIZE", "8"))
    NUM_WORKERS: int = int(os.getenv("PFI_NUM_WORKERS", "0"))
    MAX_EPOCHS: int = int(os.getenv("PFI_MAX_EPOCHS", "80"))
    EARLY_STOPPING_PATIENCE: int = int(os.getenv("PFI_EARLY_STOP_PATIENCE", "12"))
    LEARNING_RATE: float = float(os.getenv("PFI_LR", "0.0008"))
    WEIGHT_DECAY: float = float(os.getenv("PFI_WEIGHT_DECAY", "0.0001"))
    USE_AMP: bool = env_bool("PFI_USE_AMP", True)
    GRADIENT_CLIP_NORM: float = float(os.getenv("PFI_GRADIENT_CLIP_NORM", "1.0"))
    CHECKPOINT_EVERY_EPOCH: bool = env_bool("PFI_CHECKPOINT_EVERY_EPOCH", True)
    ALLOW_SPLIT_GENERATION: bool = env_bool("ALLOW_SPLIT_GENERATION", False)
    ENABLE_TRAIN_AUGMENTATION: bool = env_bool("ENABLE_TRAIN_AUGMENTATION", True)
    ENABLE_SPATIAL_AUGMENTATION: bool = env_bool("ENABLE_SPATIAL_AUGMENTATION", False)
    ENABLE_HORIZONTAL_FLIP: bool = env_bool("ENABLE_HORIZONTAL_FLIP", False)
    RESUME_MODE: Literal["off", "auto", "required"] = os.getenv("RESUME_MODE", "auto")
    AXIAL_IMAGE_COL: str = os.getenv("AXIAL_IMAGE_COL", "image_file_path")
    AXIAL_MASK_COL: str = os.getenv("AXIAL_MASK_COL", "final_label_file_path")
    AXIAL_PATIENT_COL: str = os.getenv("AXIAL_PATIENT_COL", "case_id_norm")
    AXIAL_SPLIT_COL: str = os.getenv("AXIAL_SPLIT_COL", "split")
    AXIAL_STUDY_COL: str = os.getenv("AXIAL_STUDY_COL", "case_id_norm")
    SMOKE_MAX_RECORDS_PER_SPLIT: int = int(os.getenv("SMOKE_MAX_RECORDS_PER_SPLIT", "12"))
    WEIGHT_MAX_RECORDS: int = int(os.getenv("PFI_WEIGHT_MAX_RECORDS", "256"))
    RAW0_WEIGHT_BOOST: float = float(os.getenv("AXIAL_RAW0_WEIGHT_BOOST", "1.0"))
    AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS: bool = env_bool("AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS", False)
    AXIAL_MONITOR_METRIC: Literal["dice_macro_foreground", "dice_macro_excluding_raw0"] = os.getenv("AXIAL_MONITOR_METRIC", "dice_macro_foreground")
    QUALITY_GATE_DICE_MACRO_FOREGROUND: float = float(os.getenv("QUALITY_GATE_DICE_MACRO_FOREGROUND", "0.70"))
    QUALITY_GATE_REQUIRE_PATIENT_HELDOUT: bool = True
    QUALITY_GATE_REQUIRE_RAW0_REPORT: bool = True
    QUALITY_GATE_REQUIRE_IOU: bool = True

    def __post_init__(self):
        if self.RUN_MODE != "evaluate":
            raise ValueError(f"RUN_MODE invalido: {self.RUN_MODE}")
        if self.AXIAL_MONITOR_METRIC not in {"dice_macro_foreground", "dice_macro_excluding_raw0"}:
            raise ValueError(f"AXIAL_MONITOR_METRIC invalido: {self.AXIAL_MONITOR_METRIC}")
        if self.TARGET_SIZE != (256, 256) or self.NUM_CLASSES != 6 or self.BASE_CHANNELS != 16:
            raise ValueError("Arquitectura axial final debe ser target 256, numClasses 6, baseChannels 16")


In [ ]:
CFG = TrainConfig()
MODEL_KEY = "axial_t2_alkafri"
RAW_ALLOWED_LABELS = {0, 50, 100, 150, 200, 250}
RAW_TO_CLASS_INDEX = {250: 0, 0: 1, 50: 2, 100: 3, 150: 4, 200: 5}
CLASS_INDEX_TO_NAME = {0: "background_250", 1: "raw_0", 2: "raw_50", 3: "raw_100", 4: "raw_150", 5: "raw_200"}
FOREGROUND_CLASS_IDS = [1, 2, 3, 4, 5]
FOREGROUND_EXCLUDING_RAW0 = [2, 3, 4, 5]
humanReviewRequired = True
notClinicalDiagnosis = True
OUTPUT_DIRS = {name: CFG.OUTPUT_ROOT / CFG.RUN_ID / name for name in ["models", "manifests", "metrics", "figures", "predictions", "logs", "reports"]}
RESUME_DIR = CFG.RESUME_ROOT / CFG.RUN_ID
ALLOWED_MONITOR_METRICS = {"dice_macro_foreground", "dice_macro_excluding_raw0"}
print(json.dumps(dataclasses.asdict(CFG), indent=2, default=str))


PREPROCESSING_CONFIG = {
    "normalization": "robust_percentile_1_99_per_image_train_independent",
    "imageResize": "bilinear",
    "maskResize": "nearest",
    "rawToClassIndex": RAW_TO_CLASS_INDEX,
    "augmentations": {
        "enabled": CFG.ENABLE_TRAIN_AUGMENTATION,
        "spatialEnabledByDefault": False,
        "horizontalFlip": {"enabled": CFG.ENABLE_HORIZONTAL_FLIP, "p": 0.5, "note": "requiere validar lateralidad antes de habilitar"},
        "smallRotationDegrees": {"enabled": False, "maxAbsDegrees": 5, "imageInterpolation": "bilinear", "maskInterpolation": "nearest"},
        "intensityJitter": {"enabled": True, "p": 0.25, "scale": [0.95, 1.05], "shift": [-0.03, 0.03]},
    },
}


In [ ]:
AI_SERVICE_ROOT = CFG.REPO_ROOT / "ai_service"
MODEL_ARCHITECTURES_FILE = AI_SERVICE_ROOT / "pfi_ai_service" / "model_architectures.py"

if not CFG.REPO_ROOT.exists():
    raise FileNotFoundError(
        f"CFG.REPO_ROOT no existe: {CFG.REPO_ROOT}. "
        "Ejecuta primero la celda que prepara el repositorio local."
    )

if not MODEL_ARCHITECTURES_FILE.exists():
    raise FileNotFoundError(
        f"No se encontro model_architectures.py en: {MODEL_ARCHITECTURES_FILE}"
    )

ai_service_path = str(AI_SERVICE_ROOT)
if ai_service_path not in sys.path:
    sys.path.insert(0, ai_service_path)

importlib.invalidate_caches()


In [ ]:
from pfi_ai_service.model_architectures import (
    AxialUNet2D,
    build_checkpoint_model,
    checkpoint_state_dict,
)


def build_model() -> nn.Module:
    return AxialUNet2D(
        num_classes=CFG.NUM_CLASSES,
        base_channels=CFG.BASE_CHANNELS,
    )


def strict_runtime_artifact_smoke(path: Path) -> list[int]:
    checkpoint = torch.load(
        str(path),
        map_location="cpu",
        weights_only=False,
    )

    model, _metadata = build_checkpoint_model(
        MODEL_KEY,
        checkpoint,
    )

    model.eval()

    with torch.inference_mode():
        output = model(
            torch.randn(1, 1, *CFG.TARGET_SIZE)
        )

    expected = [
        1,
        CFG.NUM_CLASSES,
        *CFG.TARGET_SIZE,
    ]

    if list(output.shape) != expected:
        raise ValueError(
            f"Shape runtime invalida: {list(output.shape)}; "
            f"esperada: {expected}"
        )

    if not torch.isfinite(output).all():
        raise ValueError(
            "El output del runtime contiene NaN o Inf"
        )

    return list(output.shape)


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def git_value(args: list[str]) -> str:
    try:
        return subprocess.check_output(["git", *args], cwd=str(CFG.REPO_ROOT), text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unavailable"


def environment_manifest() -> dict[str, Any]:
    cuda = torch.cuda.is_available()
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cudaAvailable": cuda,
        "gpuName": torch.cuda.get_device_name(0) if cuda else None,
        "gpuVramGb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda else None,
        "gitCommit": git_value(["rev-parse", "HEAD"]),
        "aiServiceCommit": AI_SERVICE_COMMIT_SHA,
        "gitDirty": bool(git_value(["status", "--short"])),
        "PFI_CLOUD_MODE": PFI_CLOUD_MODE,
        "PFI_USE_GOOGLE_DRIVE": PFI_USE_GOOGLE_DRIVE,
        "PFI_SYNC_DRY_RUN": PFI_SYNC_DRY_RUN,
    }


set_seed(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG.RUN_MODE == "train" and DEVICE.type != "cuda":
    raise RuntimeError("RUN_MODE=train requiere CUDA para evitar una corrida final accidental en CPU")
ENV_MANIFEST = environment_manifest()
print(json.dumps(ENV_MANIFEST, indent=2))


In [ ]:
def mkdirs_for_run() -> None:
    for path in [*OUTPUT_DIRS.values(), RESUME_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(handle, "wb") as fh:
            fh.write(data)
            fh.flush()
            os.fsync(fh.fileno())
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    atomic_write_bytes(path, json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8"))


def atomic_write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        atomic_write_bytes(path, b"")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    try:
        with os.fdopen(handle, "w", encoding="utf-8", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
            fh.flush()
            os.fsync(fh.fileno())
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def safe_source_name(path: str | Path) -> str:
    return Path(path).name


In [ ]:
def open_medical_array(path: Path) -> np.ndarray:
    suffixes = [s.lower() for s in path.suffixes]
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.asarray(np.load(path))
    if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return np.asarray(Image.open(path))
    if suffix in {".dcm", ".ima"}:
        if pydicom is None:
            raise RuntimeError("pydicom no disponible")
        return np.asarray(pydicom.dcmread(str(path)).pixel_array)
    if suffix in {".mha", ".mhd"}:
        if sitk is None:
            raise RuntimeError("SimpleITK no disponible")
        return sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    if suffix == ".nii" or suffixes[-2:] == [".nii", ".gz"]:
        if nib is None:
            raise RuntimeError("nibabel no disponible")
        return np.asarray(nib.load(str(path)).get_fdata())
    raise ValueError(f"Formato no soportado: {suffix}")


def as_2d_slice(array: np.ndarray, case_id: str) -> np.ndarray:
    arr = np.asarray(array)
    if arr.ndim == 2:
        return arr
    if arr.ndim == 3 and 1 in arr.shape:
        return np.squeeze(arr)
    raise ValueError(f"{case_id}: shape axial no soportado {arr.shape}")


def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.nan_to_num(np.asarray(image, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [1.0, 99.0])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(arr.min()), float(arr.max())
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return ((np.clip(arr, lo, hi) - lo) / (hi - lo)).astype(np.float32)


def resize_image(image: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(image.astype(np.float32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.BILINEAR), dtype=np.float32)


def resize_mask(mask: np.ndarray) -> np.ndarray:
    return np.asarray(Image.fromarray(mask.astype(np.int32)).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.NEAREST), dtype=np.int64)


def apply_label_mapping(mask: np.ndarray, case_id: str) -> np.ndarray:
    labels = {int(v) for v in np.unique(mask)}
    unexpected = sorted(labels - RAW_ALLOWED_LABELS)
    if unexpected:
        raise ValueError(f"{case_id}: labels inesperados {unexpected}")
    mapped = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in RAW_TO_CLASS_INDEX.items():
        mapped[np.asarray(mask) == raw_label] = class_id
    return mapped


In [ ]:
@dataclass(frozen=True)
class AxialRecord:
    image_path: str
    mask_path: str
    patientId: str
    studyId: str
    sliceId: str
    split: str
    sourceImage: str
    sourceMask: str


def resolve_path(value: Any, bases: Iterable[Path]) -> Path:
    raw = Path(str(value))
    if raw.is_absolute() and raw.exists():
        return raw
    for base in bases:
        for candidate in [base / raw, base / raw.name]:
            if candidate.exists():
                return candidate
    return raw


def required_cell_value(row: Any, column: str, row_number: int) -> str:
    value = getattr(row, column)
    if pd.isna(value):
        raise ValueError(f"Fila {row_number}: columna requerida nula: {column}")
    text = str(value).strip()
    if not text or text.lower() == "nan":
        raise ValueError(f"Fila {row_number}: columna requerida vacia/nan: {column}")
    return text


def slice_id_from_paths(patient_id: str, image_path: Path, mask_path: Path) -> str:
    material = f"{patient_id}|{image_path.resolve()}|{mask_path.resolve()}"
    return "slice_" + hashlib.sha256(material.encode("utf-8")).hexdigest()[:16]


def build_records_from_split_manifest() -> list[AxialRecord]:
    if not CFG.SPLIT_MANIFEST_PATH.exists():
        if CFG.ALLOW_SPLIT_GENERATION:
            raise NotImplementedError("Solo se permite generar candidate split con revision humana posterior")
        raise FileNotFoundError(f"Falta split manifest E9: {CFG.SPLIT_MANIFEST_PATH}")
    df = pd.read_csv(CFG.SPLIT_MANIFEST_PATH)
    required = [CFG.AXIAL_IMAGE_COL, CFG.AXIAL_MASK_COL, CFG.AXIAL_PATIENT_COL, CFG.AXIAL_SPLIT_COL]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"Faltan columnas: {missing}")
    bases = [CFG.DATASET_ROOT, CFG.IMAGES_ROOT, CFG.MASKS_ROOT, CFG.SPLIT_MANIFEST_PATH.parent]
    records = []
    for row_number, row in enumerate(df.itertuples(index=False), 2):
        image_value = required_cell_value(row, CFG.AXIAL_IMAGE_COL, row_number)
        mask_value = required_cell_value(row, CFG.AXIAL_MASK_COL, row_number)
        patient = required_cell_value(row, CFG.AXIAL_PATIENT_COL, row_number)
        split = required_cell_value(row, CFG.AXIAL_SPLIT_COL, row_number).lower()
        study = required_cell_value(row, CFG.AXIAL_STUDY_COL, row_number) if hasattr(row, CFG.AXIAL_STUDY_COL) else patient
        image = resolve_path(image_value, bases)
        mask = resolve_path(mask_value, bases)
        if not image.exists() or not mask.exists():
            raise FileNotFoundError(f"Archivo axial inexistente: {image.name} / {mask.name}")
        records.append(AxialRecord(str(image), str(mask), patient, study, slice_id_from_paths(patient, image, mask), split, safe_source_name(image), safe_source_name(mask)))
    return records


def save_split_snapshot(records: list[AxialRecord]) -> Path:
    snapshot_path = OUTPUT_DIRS["manifests"] / "split_snapshot.csv"
    snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([dataclasses.asdict(r) for r in records]).to_csv(snapshot_path, index=False)
    return snapshot_path


def summarize_records(records: list[AxialRecord]) -> pd.DataFrame:
    df = pd.DataFrame([dataclasses.asdict(r) for r in records])
    return df.groupby("split").agg(n_slices=("sliceId", "count"), n_patients=("patientId", "nunique"))


In [ ]:
def validate_split_integrity(records: list[AxialRecord]) -> dict[str, Any]:
    df = pd.DataFrame([dataclasses.asdict(r) for r in records])
    if df.empty:
        raise ValueError("Split manifest sin filas")
    bad = sorted(set(df["split"]) - {"train", "val", "test"})
    if bad:
        raise ValueError(f"Splits invalidos: {bad}")
    for split in ["train", "val", "test"]:
        if int((df["split"] == split).sum()) == 0:
            raise ValueError(f"Split vacio: {split}")
    duplicate_rows = int(df.duplicated(subset=["image_path", "mask_path", "patientId", "split"]).sum())
    if duplicate_rows:
        raise ValueError(f"Filas exactamente duplicadas: {duplicate_rows}")
    leakage_by_key = {}
    for key in ["patientId", "studyId", "image_path", "mask_path"]:
        leaked = df.groupby(key)["split"].nunique()
        leaked = leaked[leaked > 1]
        leakage_by_key[key] = int(len(leaked))
        if not leaked.empty:
            raise ValueError(f"Leakage cross-split en {key}: {len(leaked)}")
    missing_paths = [r.sourceImage for r in records if not Path(r.image_path).exists()]
    missing_paths += [r.sourceMask for r in records if not Path(r.mask_path).exists()]
    if missing_paths:
        raise FileNotFoundError(f"Paths faltantes: {missing_paths[:20]}")
    return {
        "patientHeldout": True,
        "leakageDetected": False,
        "duplicateRows": duplicate_rows,
        "leakageByKey": leakage_by_key,
        "splitCounts": df["split"].value_counts().to_dict(),
        "patientsBySplit": df.groupby("split")["patientId"].nunique().to_dict(),
        "imagesBySplit": df.groupby("split")["image_path"].nunique().to_dict(),
    }


def scan_labels_and_shapes(records: list[AxialRecord]) -> dict[str, Any]:
    pixel_counts: dict[str, Counter[int]] = defaultdict(Counter)
    case_presence: dict[str, Counter[int]] = defaultdict(Counter)
    raw0_by_split: dict[str, set[str]] = defaultdict(set)
    formats: Counter[str] = Counter()
    dimensions: Counter[str] = Counter()
    shape_mismatches: list[dict[str, Any]] = []
    shape_warnings: list[str] = []

    for index, record in enumerate(records, 1):
        image = as_2d_slice(
            open_medical_array(Path(record.image_path)),
            record.sliceId,
        )
        mask = as_2d_slice(
            open_medical_array(Path(record.mask_path)),
            record.sliceId,
        )

        if image.shape != mask.shape:
            image_ratio = image.shape[1] / image.shape[0]
            mask_ratio = mask.shape[1] / mask.shape[0]

            mismatch = {
                "sliceId": record.sliceId,
                "patientId": record.patientId,
                "studyId": record.studyId,
                "split": record.split,
                "imageShape": list(image.shape),
                "maskShape": list(mask.shape),
                "imageAspectRatio": float(image_ratio),
                "maskAspectRatio": float(mask_ratio),
                "sourceImage": record.sourceImage,
                "sourceMask": record.sourceMask,
            }

            shape_mismatches.append(mismatch)

            known_pattern = (
                str(record.patientId) == "56"
                and tuple(image.shape) == (384, 384)
                and tuple(mask.shape) == (320, 320)
                and np.isclose(
                    image_ratio,
                    mask_ratio,
                    rtol=0,
                    atol=1e-6,
                )
            )

            if not known_pattern:
                raise ValueError(
                    f"{record.sliceId}: mismatch image/mask no autorizado: "
                    f"{image.shape} != {mask.shape}"
                )

            message = (
                f"{record.sliceId}: patron conocido del paciente 56: "
                f"imagen {image.shape}, mascara {mask.shape}; "
                "se normalizan ambas a 256x256. "
                "Alineacion visual revisada; diferencia registrada."
            )

            warnings.warn(message)
            shape_warnings.append(message)

        labels = {int(value) for value in np.unique(mask)}
        unexpected = sorted(labels - RAW_ALLOWED_LABELS)
        if unexpected:
            raise ValueError(
                f"{record.sliceId}: labels inesperados {unexpected}"
            )

        mapped = apply_label_mapping(mask, record.sliceId)
        bincount = np.bincount(
            mapped.reshape(-1),
            minlength=CFG.NUM_CLASSES,
        )

        for class_id, count in enumerate(bincount):
            pixel_counts[record.split][class_id] += int(count)
            if count > 0:
                case_presence[record.split][class_id] += 1

        if bincount[1] > 0:
            raw0_by_split[record.split].add(record.patientId)

        formats[Path(record.image_path).suffix.lower()] += 1
        dimensions[f"image={tuple(image.shape)}|mask={tuple(mask.shape)}"] += 1

        if index % 250 == 0:
            print(f"scan {index}/{len(records)}")

    absent_train = [
        CLASS_INDEX_TO_NAME[class_id]
        for class_id in range(CFG.NUM_CLASSES)
        if case_presence.get("train", Counter()).get(class_id, 0) == 0
    ]

    if absent_train:
        raise ValueError(
            f"Clases ausentes en train: {absent_train}"
        )

    warnings_by_split: list[str] = []
    for split in ["val", "test"]:
        absent = [
            CLASS_INDEX_TO_NAME[class_id]
            for class_id in range(CFG.NUM_CLASSES)
            if case_presence.get(split, Counter()).get(class_id, 0) == 0
        ]
        if absent:
            message = f"WARNING: clases ausentes en {split}: {absent}"
            warnings.warn(message)
            warnings_by_split.append(message)

    return {
        "pixelCountsBySplit": {
            split: {
                CLASS_INDEX_TO_NAME[class_id]: int(count)
                for class_id, count in counter.items()
            }
            for split, counter in pixel_counts.items()
        },
        "casePresenceBySplit": {
            split: {
                CLASS_INDEX_TO_NAME[class_id]: int(count)
                for class_id, count in counter.items()
            }
            for split, counter in case_presence.items()
        },
        "raw0PatientsBySplit": {
            split: len(patient_ids)
            for split, patient_ids in raw0_by_split.items()
        },
        "formats": dict(formats),
        "dimensions": dict(dimensions),
        "shapeMismatchCount": len(shape_mismatches),
        "shapeMismatches": shape_mismatches,
        "shapeWarnings": shape_warnings,
        "warnings": warnings_by_split + shape_warnings,
    }


In [ ]:
AUGMENTATION_EPOCH = 0


def set_augmentation_epoch(epoch: int) -> None:
    global AUGMENTATION_EPOCH
    AUGMENTATION_EPOCH = int(epoch)


def maybe_augment(image: np.ndarray, mask: np.ndarray, rng: random.Random) -> tuple[np.ndarray, np.ndarray]:
    if not CFG.ENABLE_TRAIN_AUGMENTATION:
        return image, mask
    if CFG.ENABLE_SPATIAL_AUGMENTATION and CFG.ENABLE_HORIZONTAL_FLIP and rng.random() < 0.5:
        image, mask = np.flip(image, 1).copy(), np.flip(mask, 1).copy()
    if rng.random() < 0.25:
        image = np.clip(image * rng.uniform(0.95, 1.05) + rng.uniform(-0.03, 0.03), 0, 1).astype(np.float32)
    return image, mask


class AxialSegmentationDataset(Dataset):
    def __init__(self, records: list[AxialRecord], split: str, augment: bool):
        self.records = [r for r in records if r.split == split]
        self.augment = augment
        if not self.records:
            raise ValueError(f"Dataset vacio: {split}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict[str, Any]:
        record = self.records[index]
        image = resize_image(robust_normalize(as_2d_slice(open_medical_array(Path(record.image_path)), record.sliceId)))
        mask = apply_label_mapping(resize_mask(as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)), record.sliceId)
        if self.augment:
            seed_material = f"{CFG.SEED}|{AUGMENTATION_EPOCH}|{index}|{record.sliceId}"
            seed = int(hashlib.sha256(seed_material.encode("utf-8")).hexdigest()[:8], 16)
            image, mask = maybe_augment(image, mask, random.Random(seed))
        return {"image": torch.from_numpy(image[None].astype(np.float32)), "mask": torch.from_numpy(mask.astype(np.int64)), "patientId": record.patientId, "studyId": record.studyId, "sliceId": record.sliceId, "sourceImage": record.sourceImage, "sourceMask": record.sourceMask}


def limit_records_for_smoke(records: list[AxialRecord]) -> list[AxialRecord]:
    selected: list[AxialRecord] = []
    for split in ["test"]:
        split_records = [r for r in records if r.split == split]
        by_patient: dict[str, list[AxialRecord]] = defaultdict(list)
        for record in split_records:
            by_patient[record.patientId].append(record)
        patient_ids = sorted(by_patient)[: max(2, min(2, len(by_patient)))]
        for patient_id in patient_ids:
            selected.extend(by_patient[patient_id][: max(1, CFG.SMOKE_MAX_RECORDS_PER_SPLIT // max(len(patient_ids), 1))])
    summary = pd.DataFrame([dataclasses.asdict(r) for r in selected])
    if not summary.empty:
        print("smoke selection", summary.groupby("split").agg(patients=("patientId", "nunique"), slices=("sliceId", "count")).to_dict())
    return selected


def seed_worker(worker_id: int) -> None:
    random.seed(CFG.SEED + worker_id)
    np.random.seed(CFG.SEED + worker_id)


def build_eval_loaders(records: list[AxialRecord], smoke: bool) -> dict[str, DataLoader]:
    if smoke:
        records = limit_records_for_smoke(records)
    generator = torch.Generator().manual_seed(CFG.SEED)
    return {split: DataLoader(AxialSegmentationDataset(records, split, split == "train"), batch_size=CFG.BATCH_SIZE, shuffle=split == "train", num_workers=CFG.NUM_WORKERS, worker_init_fn=seed_worker, generator=generator, pin_memory=DEVICE.type == "cuda", persistent_workers=CFG.NUM_WORKERS > 0, drop_last=False) for split in ["test"]}



def build_loaders(records: list[AxialRecord], smoke: bool) -> dict[str, DataLoader]:
    """Compatibilidad para tests sinteticos: notebook 49 solo devuelve train/val."""
    return build_eval_loaders(records, smoke)


In [ ]:
def class_weight_report(records: list[AxialRecord]) -> dict[str, Any]:
    counts = np.ones(CFG.NUM_CLASSES, dtype=np.float64)
    image_presence = np.zeros(CFG.NUM_CLASSES, dtype=np.int64)
    train_records = [r for r in records if r.split == "train"][: CFG.WEIGHT_MAX_RECORDS]
    for record in train_records:
        mapped = apply_label_mapping(resize_mask(as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)), record.sliceId)
        bincount = np.bincount(mapped.reshape(-1), minlength=CFG.NUM_CLASSES)
        counts += bincount
        image_presence += (bincount > 0).astype(np.int64)
    freq = counts / counts.sum()
    base = 1.0 / np.sqrt(freq)
    base = base / base.mean()
    boosted = base.copy()
    boosted[1] *= CFG.RAW0_WEIGHT_BOOST
    final = boosted / boosted.mean()
    ratio = float(final.max() / max(final.min(), 1e-8))
    if ratio > 12:
        warnings.warn(f"WARNING: class weight max/min desproporcionado: {ratio:.2f}")
    if ratio > 40 and not CFG.AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS:
        raise ValueError(f"class weight max/min extremo: {ratio:.2f}")
    return {
        "trainSamplesUsed": len(train_records),
        "pixelCounts": {CLASS_INDEX_TO_NAME[i]: int(counts[i]) for i in range(CFG.NUM_CLASSES)},
        "imagePresence": {CLASS_INDEX_TO_NAME[i]: int(image_presence[i]) for i in range(CFG.NUM_CLASSES)},
        "baseWeights": {CLASS_INDEX_TO_NAME[i]: float(base[i]) for i in range(CFG.NUM_CLASSES)},
        "raw0Boost": CFG.RAW0_WEIGHT_BOOST,
        "finalWeights": {CLASS_INDEX_TO_NAME[i]: float(final[i]) for i in range(CFG.NUM_CLASSES)},
        "maxMinRatio": ratio,
    }


def estimate_class_weights(records: list[AxialRecord]) -> torch.Tensor:
    report = class_weight_report(records)
    return torch.tensor([report["finalWeights"][CLASS_INDEX_TO_NAME[i]] for i in range(CFG.NUM_CLASSES)], dtype=torch.float32)


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor, include_background: bool = False, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(target, CFG.NUM_CLASSES).permute(0, 3, 1, 2).float()
    class_ids = range(CFG.NUM_CLASSES) if include_background else FOREGROUND_CLASS_IDS
    losses = []
    for class_id in class_ids:
        p, t = probs[:, class_id], one_hot[:, class_id]
        losses.append(1 - ((2 * (p * t).sum((1, 2)) + eps) / (p.sum((1, 2)) + t.sum((1, 2)) + eps)))
    return torch.stack(losses).mean()


def multiclass_loss(logits: torch.Tensor, target: torch.Tensor, ce: nn.Module) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ce_loss = ce(logits, target)
    dice_loss_value = soft_dice_loss(logits, target)
    return ce_loss + dice_loss_value, ce_loss, dice_loss_value


def dice_iou_for_class(pred: np.ndarray, truth: np.ndarray, class_id: int) -> tuple[float | None, float | None, int, int, int]:
    pred_mask, truth_mask = pred == class_id, truth == class_id
    tp = int(np.logical_and(pred_mask, truth_mask).sum())
    fp = int(np.logical_and(pred_mask, ~truth_mask).sum())
    fn = int(np.logical_and(~pred_mask, truth_mask).sum())
    dice_den, iou_den = 2 * tp + fp + fn, tp + fp + fn
    # Si GT y pred no tienen la clase, la metrica queda indefinida y se excluye del macro.
    return (None if dice_den == 0 else 2 * tp / dice_den, None if iou_den == 0 else tp / iou_den, int(truth_mask.any()), int(pred_mask.any()), int(not truth_mask.any()))


def metrics_from_predictions(pred: np.ndarray, truth: np.ndarray) -> dict[str, Any]:
    per_class = {}
    confusion = np.zeros((CFG.NUM_CLASSES, CFG.NUM_CLASSES), dtype=np.int64)
    for p, t in zip(pred, truth):
        valid = (t >= 0) & (t < CFG.NUM_CLASSES)
        confusion += np.bincount(CFG.NUM_CLASSES * t[valid].reshape(-1) + p[valid].reshape(-1), minlength=CFG.NUM_CLASSES**2).reshape(CFG.NUM_CLASSES, CFG.NUM_CLASSES)
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        values = [dice_iou_for_class(p, t, class_id) for p, t in zip(pred, truth)]
        dice = [v[0] for v in values if v[0] is not None]
        iou = [v[1] for v in values if v[1] is not None]
        per_class[name] = {
            "dice": float(np.mean(dice)) if dice else None,
            "iou": float(np.mean(iou)) if iou else None,
            "evaluable_cases": len(dice),
            "gt_present_cases": sum(v[2] for v in values),
            "pred_present_cases": sum(v[3] for v in values),
            "gt_absent_cases": sum(v[4] for v in values),
        }
    def macro(class_ids: list[int], metric: str) -> float | None:
        vals = [per_class[CLASS_INDEX_TO_NAME[c]][metric] for c in class_ids if per_class[CLASS_INDEX_TO_NAME[c]][metric] is not None]
        return float(np.mean(vals)) if vals else None
    return {
        "perClass": per_class,
        "dice_macro_foreground": macro(FOREGROUND_CLASS_IDS, "dice"),
        "iou_macro_foreground": macro(FOREGROUND_CLASS_IDS, "iou"),
        "dice_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "dice"),
        "iou_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "iou"),
        "raw0Dice": per_class["raw_0"]["dice"],
        "raw0Iou": per_class["raw_0"]["iou"],
        "raw0Precision": None if int(confusion[:, 1].sum()) == 0 else float(confusion[1, 1] / max(int(confusion[:, 1].sum()), 1)),
        "raw0Recall": None if int(confusion[1, :].sum()) == 0 else float(confusion[1, 1] / max(int(confusion[1, :].sum()), 1)),
        "raw0PredictedInGtAbsentCases": max(0, per_class["raw_0"]["pred_present_cases"] - per_class["raw_0"]["gt_present_cases"]),
        "confusionMatrix": confusion.tolist(),
    }



def metrics_from_logits(logits: torch.Tensor, target: torch.Tensor) -> dict[str, Any]:
    pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
    truth = target.detach().cpu().numpy()
    return metrics_from_predictions(pred, truth)


In [ ]:
def save_checkpoint(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    os.close(handle)
    try:
        torch.save(payload, tmp_name)
        Path(tmp_name).replace(path)
    except Exception:
        Path(tmp_name).unlink(missing_ok=True)
        raise


def split_sha256() -> str:
    return sha256_stream(CFG.SPLIT_MANIFEST_PATH) if CFG.SPLIT_MANIFEST_PATH.exists() else "missing"


In [ ]:
def class_weight_report(records: list[AxialRecord]) -> dict[str, Any]:
    counts = np.ones(CFG.NUM_CLASSES, dtype=np.float64)
    image_presence = np.zeros(CFG.NUM_CLASSES, dtype=np.int64)
    train_records = [r for r in records if r.split == "train"][: CFG.WEIGHT_MAX_RECORDS]
    for record in train_records:
        mapped = apply_label_mapping(resize_mask(as_2d_slice(open_medical_array(Path(record.mask_path)), record.sliceId)), record.sliceId)
        bincount = np.bincount(mapped.reshape(-1), minlength=CFG.NUM_CLASSES)
        counts += bincount
        image_presence += (bincount > 0).astype(np.int64)
    freq = counts / counts.sum()
    base = 1.0 / np.sqrt(freq)
    base = base / base.mean()
    boosted = base.copy()
    boosted[1] *= CFG.RAW0_WEIGHT_BOOST
    final = boosted / boosted.mean()
    ratio = float(final.max() / max(final.min(), 1e-8))
    if ratio > 12:
        warnings.warn(f"WARNING: class weight max/min desproporcionado: {ratio:.2f}")
    if ratio > 40 and not CFG.AXIAL_ALLOW_EXTREME_CLASS_WEIGHTS:
        raise ValueError(f"class weight max/min extremo: {ratio:.2f}")
    return {
        "trainSamplesUsed": len(train_records),
        "pixelCounts": {CLASS_INDEX_TO_NAME[i]: int(counts[i]) for i in range(CFG.NUM_CLASSES)},
        "imagePresence": {CLASS_INDEX_TO_NAME[i]: int(image_presence[i]) for i in range(CFG.NUM_CLASSES)},
        "baseWeights": {CLASS_INDEX_TO_NAME[i]: float(base[i]) for i in range(CFG.NUM_CLASSES)},
        "raw0Boost": CFG.RAW0_WEIGHT_BOOST,
        "finalWeights": {CLASS_INDEX_TO_NAME[i]: float(final[i]) for i in range(CFG.NUM_CLASSES)},
        "maxMinRatio": ratio,
    }


def estimate_class_weights(records: list[AxialRecord]) -> torch.Tensor:
    report = class_weight_report(records)
    return torch.tensor([report["finalWeights"][CLASS_INDEX_TO_NAME[i]] for i in range(CFG.NUM_CLASSES)], dtype=torch.float32)


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor, include_background: bool = False, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(target, CFG.NUM_CLASSES).permute(0, 3, 1, 2).float()
    class_ids = range(CFG.NUM_CLASSES) if include_background else FOREGROUND_CLASS_IDS
    losses = []
    for class_id in class_ids:
        p, t = probs[:, class_id], one_hot[:, class_id]
        losses.append(1 - ((2 * (p * t).sum((1, 2)) + eps) / (p.sum((1, 2)) + t.sum((1, 2)) + eps)))
    return torch.stack(losses).mean()


def multiclass_loss(logits: torch.Tensor, target: torch.Tensor, ce: nn.Module) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ce_loss = ce(logits, target)
    dice_loss_value = soft_dice_loss(logits, target)
    return ce_loss + dice_loss_value, ce_loss, dice_loss_value


def _optional_div(numerator: int, denominator: int) -> float | None:
    return None if denominator == 0 else float(numerator / denominator)


def _case_metric_row(record: AxialRecord, pred: np.ndarray, truth: np.ndarray) -> dict[str, Any]:
    per_class = {}
    gt_classes = sorted(int(v) for v in np.unique(truth) if 0 <= int(v) < CFG.NUM_CLASSES)
    pred_classes = sorted(int(v) for v in np.unique(pred) if 0 <= int(v) < CFG.NUM_CLASSES)
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        pred_mask = pred == class_id
        truth_mask = truth == class_id
        tp = int(np.logical_and(pred_mask, truth_mask).sum())
        fp = int(np.logical_and(pred_mask, ~truth_mask).sum())
        fn = int(np.logical_and(~pred_mask, truth_mask).sum())
        dice_den = 2 * tp + fp + fn
        iou_den = tp + fp + fn
        per_class[name] = {
            "dice": _optional_div(2 * tp, dice_den),
            "iou": _optional_div(tp, iou_den),
            "gt_present": bool(truth_mask.any()),
            "pred_present": bool(pred_mask.any()),
            "gt_absent": not bool(truth_mask.any()),
            "predicted_in_gt_absent": int(bool(pred_mask.any()) and not bool(truth_mask.any())),
        }
    def macro(class_ids: list[int], metric: str) -> float | None:
        vals = [per_class[CLASS_INDEX_TO_NAME[c]][metric] for c in class_ids if per_class[CLASS_INDEX_TO_NAME[c]][metric] is not None]
        return float(np.mean(vals)) if vals else None
    raw0 = per_class["raw_0"]
    row = {
        "patientId": record.patientId,
        "studyId": record.studyId,
        "sliceId": record.sliceId,
        "sourceImage": record.sourceImage,
        "sourceMask": record.sourceMask,
        "gtClasses": json.dumps(gt_classes),
        "predClasses": json.dumps(pred_classes),
        "diceMacroForeground": macro(FOREGROUND_CLASS_IDS, "dice"),
        "iouMacroForeground": macro(FOREGROUND_CLASS_IDS, "iou"),
        "diceMacroExcludingRaw0": macro(FOREGROUND_EXCLUDING_RAW0, "dice"),
        "raw0GtPresent": raw0["gt_present"],
        "raw0PredPresent": raw0["pred_present"],
        "raw0PredictedInGtAbsent": raw0["predicted_in_gt_absent"],
        "raw0Dice": raw0["dice"],
        "raw0Iou": raw0["iou"],
        "raw0Precision": None,
        "raw0Recall": None,
    }
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        row[f"dice_{name}"] = per_class[name]["dice"]
    return row


def metrics_from_predictions(pred: np.ndarray, truth: np.ndarray) -> dict[str, Any]:
    if pred.shape != truth.shape:
        raise ValueError(f"Shape mismatch pred/truth: {pred.shape} != {truth.shape}")
    confusion = np.zeros((CFG.NUM_CLASSES, CFG.NUM_CLASSES), dtype=np.int64)
    per_class_counts = {
        name: {
            "evaluableCases": 0,
            "gtPresentCases": 0,
            "predPresentCases": 0,
            "gtAbsentCases": 0,
            "predictedInGtAbsentCases": 0,
        }
        for name in CLASS_INDEX_TO_NAME.values()
    }
    for p, t in zip(pred, truth):
        valid = (t >= 0) & (t < CFG.NUM_CLASSES)
        confusion += np.bincount(CFG.NUM_CLASSES * t[valid].reshape(-1) + p[valid].reshape(-1), minlength=CFG.NUM_CLASSES**2).reshape(CFG.NUM_CLASSES, CFG.NUM_CLASSES)
        pred_has_raw0 = bool(np.any(p == 1))
        gt_has_raw0 = bool(np.any(t == 1))
        raw0_predicted_in_gt_absent = int(pred_has_raw0 and not gt_has_raw0)
        for class_id, name in CLASS_INDEX_TO_NAME.items():
            pred_has = pred_has_raw0 if class_id == 1 else bool(np.any(p == class_id))
            gt_has = gt_has_raw0 if class_id == 1 else bool(np.any(t == class_id))
            per_class_counts[name]["evaluableCases"] += 1
            per_class_counts[name]["gtPresentCases"] += int(gt_has)
            per_class_counts[name]["predPresentCases"] += int(pred_has)
            per_class_counts[name]["gtAbsentCases"] += int(not gt_has)
            per_class_counts[name]["predictedInGtAbsentCases"] += raw0_predicted_in_gt_absent if class_id == 1 else int(pred_has and not gt_has)
    return metrics_from_confusion(confusion.tolist(), per_class_counts)


def metrics_from_confusion(confusion: list[list[int]], class_counts: dict[str, dict[str, int]]) -> dict[str, Any]:
    cm = np.asarray(confusion, dtype=np.int64)
    total = int(cm.sum())
    per_class = {}
    for class_id, name in CLASS_INDEX_TO_NAME.items():
        tp = int(cm[class_id, class_id])
        fp = int(cm[:, class_id].sum() - tp)
        fn = int(cm[class_id, :].sum() - tp)
        tn = int(total - tp - fp - fn)
        dice_den = 2 * tp + fp + fn
        iou_den = tp + fp + fn
        precision_den = tp + fp
        recall_den = tp + fn
        counts = class_counts.get(name, {})
        per_class[name] = {
            "dice": _optional_div(2 * tp, dice_den),
            "iou": _optional_div(tp, iou_den),
            "precision": _optional_div(tp, precision_den),
            "recall": _optional_div(tp, recall_den),
            "truePositivePixels": tp,
            "falsePositivePixels": fp,
            "falseNegativePixels": fn,
            "trueNegativePixels": tn,
            "evaluableCases": int(counts.get("evaluableCases", 0)),
            "gtPresentCases": int(counts.get("gtPresentCases", 0)),
            "predPresentCases": int(counts.get("predPresentCases", 0)),
            "gtAbsentCases": int(counts.get("gtAbsentCases", 0)),
            "predictedInGtAbsentCases": int(counts.get("predictedInGtAbsentCases", 0)),
        }
    def macro(class_ids: list[int], metric: str) -> float | None:
        vals = [per_class[CLASS_INDEX_TO_NAME[c]][metric] for c in class_ids if per_class[CLASS_INDEX_TO_NAME[c]][metric] is not None]
        return float(np.mean(vals)) if vals else None
    raw0 = per_class["raw_0"]
    metrics = {
        "perClass": per_class,
        "dice_macro_foreground": macro(FOREGROUND_CLASS_IDS, "dice"),
        "iou_macro_foreground": macro(FOREGROUND_CLASS_IDS, "iou"),
        "precision_macro_foreground": macro(FOREGROUND_CLASS_IDS, "precision"),
        "recall_macro_foreground": macro(FOREGROUND_CLASS_IDS, "recall"),
        "dice_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "dice"),
        "iou_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "iou"),
        "precision_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "precision"),
        "recall_macro_excluding_raw0": macro(FOREGROUND_EXCLUDING_RAW0, "recall"),
        "evaluableClassCounts": {name: values["evaluableCases"] for name, values in per_class.items()},
        "raw0Dice": raw0["dice"],
        "raw0Iou": raw0["iou"],
        "raw0Precision": raw0["precision"],
        "raw0Recall": raw0["recall"],
        "raw0TruePositivePixels": raw0["truePositivePixels"],
        "raw0FalsePositivePixels": raw0["falsePositivePixels"],
        "raw0FalseNegativePixels": raw0["falseNegativePixels"],
        "raw0TrueNegativePixels": raw0["trueNegativePixels"],
        "raw0GtPresentCases": raw0["gtPresentCases"],
        "raw0PredPresentCases": raw0["predPresentCases"],
        "raw0GtAbsentCases": raw0["gtAbsentCases"],
        "raw0PredictedInGtAbsentCases": raw0["predictedInGtAbsentCases"],
        "confusionMatrix": cm.tolist(),
    }
    return metrics


def metrics_from_logits(logits: torch.Tensor, target: torch.Tensor) -> dict[str, Any]:
    pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
    truth = target.detach().cpu().numpy()
    return metrics_from_predictions(pred, truth)


def per_class_rows(metrics: dict[str, Any]) -> list[dict[str, Any]]:
    return [{"className": name, **values} for name, values in metrics.get("perClass", {}).items()]


def confusion_matrix_csv(path: Path, metrics: dict[str, Any]) -> None:
    pd.DataFrame(metrics["confusionMatrix"], index=[CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)], columns=[CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)]).to_csv(path)


def _merge_confusion(items: list[dict[str, Any]]) -> list[list[int]]:
    matrix = np.zeros((CFG.NUM_CLASSES, CFG.NUM_CLASSES), dtype=np.int64)
    for item in items:
        matrix += np.asarray(item["confusionMatrix"], dtype=np.int64)
    return matrix.tolist()


def _merge_class_counts(items: list[dict[str, Any]]) -> dict[str, dict[str, int]]:
    totals = {name: {"evaluableCases": 0, "gtPresentCases": 0, "predPresentCases": 0, "gtAbsentCases": 0, "predictedInGtAbsentCases": 0} for name in CLASS_INDEX_TO_NAME.values()}
    for item in items:
        for name, values in item.get("perClass", {}).items():
            if name not in totals:
                continue
            for key in totals[name]:
                totals[name][key] += int(values.get(key, 0))
    return totals


def run_eval_epoch(model: nn.Module, loader: DataLoader, ce: nn.Module) -> dict[str, Any]:
    model.eval()
    losses = ce_losses = dice_losses = 0.0
    metric_items = []
    with torch.inference_mode():
        for batch in loader:
            image, mask = batch["image"].to(DEVICE), batch["mask"].to(DEVICE)
            logits = model(image)
            if not torch.isfinite(logits).all():
                raise FloatingPointError("logits contiene NaN/Inf")
            loss, ce_loss, dice_loss_value = multiclass_loss(logits, mask, ce)
            for name, value in [("loss", loss), ("ce_loss", ce_loss), ("dice_loss_value", dice_loss_value)]:
                if not torch.isfinite(value):
                    raise FloatingPointError(f"{name} contiene NaN/Inf")
            losses += float(loss.detach().cpu())
            ce_losses += float(ce_loss.detach().cpu())
            dice_losses += float(dice_loss_value.detach().cpu())
            metric_items.append(metrics_from_logits(logits, mask))
    n = max(len(loader), 1)
    metrics = metrics_from_confusion(_merge_confusion(metric_items), _merge_class_counts(metric_items))
    metrics.update({"loss": losses / n, "cross_entropy": ce_losses / n, "dice_loss": dice_losses / n})
    return metrics


In [ ]:
TEST_EVALUATED_IN_MEMORY = False


def checkpoint_compatibility_errors(checkpoint: dict[str, Any], smoke_only: bool) -> list[str]:
    raw0_boost = checkpoint.get("raw0Boost")
    try:
        raw0_ok = math.isclose(float(raw0_boost), CFG.RAW0_WEIGHT_BOOST, rel_tol=0.0, abs_tol=1e-12)
    except Exception:
        raw0_ok = False
    checks = {
        "runId": checkpoint.get("runId") == CFG.RUN_ID,
        "splitSha256": checkpoint.get("splitSha256") == split_sha256(),
        "architecture": checkpoint.get("architecture") == "AxialUNet2D",
        "target_size": tuple(checkpoint.get("target_size", ())) == CFG.TARGET_SIZE,
        "num_classes": checkpoint.get("num_classes") == CFG.NUM_CLASSES,
        "base_channels": checkpoint.get("base_channels") == CFG.BASE_CHANNELS,
        "rawToClassIndex": checkpoint.get("rawToClassIndex", checkpoint.get("labelMapping", {}).get("rawToClassIndex")) == RAW_TO_CLASS_INDEX,
        "preprocessingConfig": checkpoint.get("preprocessingConfig") == PREPROCESSING_CONFIG,
        "smokeOnly": bool(checkpoint.get("smokeOnly", checkpoint.get("smoke_only", False))) is False,
        "monitorMetric": checkpoint.get("monitorMetric") == CFG.AXIAL_MONITOR_METRIC,
        "raw0Boost": raw0_ok,
        "aiServiceCommit": checkpoint.get("aiServiceCommit") == AI_SERVICE_COMMIT_SHA,
    }
    return [name for name, ok in checks.items() if not ok]


def _confirmation_hash() -> str:
    token = os.getenv("AXIAL_FINAL_TEST_CONFIRMATION", "")
    return hashlib.sha256(token.encode("utf-8")).hexdigest() if token else ""


def _predict_case_rows(model: nn.Module, loader: DataLoader) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            logits = model(batch["image"].to(DEVICE))
            if not torch.isfinite(logits).all():
                raise FloatingPointError("logits contiene NaN/Inf")
            pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
            truth = batch["mask"].detach().cpu().numpy()
            for index in range(pred.shape[0]):
                record = AxialRecord("", "", str(batch["patientId"][index]), str(batch["studyId"][index]), str(batch["sliceId"][index]), "test", str(batch["sourceImage"][index]), str(batch["sourceMask"][index]))
                rows.append(_case_metric_row(record, pred[index], truth[index]))
    return rows


def save_test_predictions_preview(model: nn.Module, loader: DataLoader, figure_path: Path, max_cases: int = 12) -> Path:
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    selected = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            logits = model(batch["image"].to(DEVICE))
            if not torch.isfinite(logits).all():
                raise FloatingPointError("logits contiene NaN/Inf")
            pred = torch.argmax(logits, dim=1).detach().cpu().numpy()
            images = batch["image"][:, 0].detach().cpu().numpy()
            masks = batch["mask"].detach().cpu().numpy()
            for i in range(pred.shape[0]):
                selected.append((images[i], masks[i], pred[i], str(batch["sliceId"][i])))
                if len(selected) >= max_cases:
                    break
            if len(selected) >= max_cases:
                break
    if not selected:
        raise ValueError("No hay casos para preview de test")
    fig, axes = plt.subplots(len(selected), 3, figsize=(9, 3 * len(selected)), squeeze=False)
    for row, (image, mask, pred, slice_id) in enumerate(selected):
        axes[row][0].imshow(image, cmap="gray")
        axes[row][0].set_title(f"{slice_id} image")
        axes[row][1].imshow(mask, cmap="tab10", vmin=0, vmax=CFG.NUM_CLASSES - 1)
        axes[row][1].set_title("GT")
        axes[row][2].imshow(pred, cmap="tab10", vmin=0, vmax=CFG.NUM_CLASSES - 1)
        axes[row][2].set_title("prediction")
        for ax in axes[row]:
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(figure_path, dpi=120)
    plt.close(fig)
    return figure_path


def evaluate_test_once(records: list[AxialRecord]) -> dict[str, Any]:
    global TEST_EVALUATED_IN_MEMORY
    marker = OUTPUT_DIRS["metrics"] / "test_evaluated_once.json"
    in_progress = OUTPUT_DIRS["metrics"] / "test_evaluation_in_progress.json"
    best_checkpoint_path = RESUME_DIR / "axial_t2_alkafri_v2.best_checkpoint.pt"
    if TEST_EVALUATED_IN_MEMORY:
        raise RuntimeError("El test ya fue evaluado en memoria para este run")
    if os.getenv("AXIAL_FINAL_TEST_CONFIRMATION") != CFG.RUN_ID:
        raise RuntimeError("AXIAL_FINAL_TEST_CONFIRMATION=axial-final-v2 requerido antes de tocar test")
    if in_progress.exists():
        raise RuntimeError("Existe test_evaluation_in_progress.json; revisar ejecucion previa antes de reintentar")
    if marker.exists():
        raise RuntimeError("test_evaluated_once.json ya existe para este run")
    checkpoint_sha = sha256_stream(best_checkpoint_path)
    checkpoint = torch.load(str(best_checkpoint_path), map_location=DEVICE, weights_only=False)
    errors = checkpoint_compatibility_errors(checkpoint, smoke_only=False)
    if errors:
        raise ValueError("Checkpoint incompatible para test-once: " + ",".join(errors))
    atomic_write_json(in_progress, {"runId": CFG.RUN_ID, "status": "in_progress", "startedAtUtc": utc_now(), "checkpointSha256": checkpoint_sha, "splitSha256": split_sha256(), "confirmationTokenHash": _confirmation_hash()})
    loaders = build_eval_loaders(records, False)
    model = build_model().to(DEVICE)
    model.load_state_dict(checkpoint_state_dict(checkpoint), strict=True)
    model.eval()
    ce = nn.CrossEntropyLoss(weight=torch.tensor(checkpoint["class_weights"], dtype=torch.float32, device=DEVICE))
    result = run_eval_epoch(model, loaders["test"], ce)
    case_rows = _predict_case_rows(model, loaders["test"])
    result.update({"testEvaluatedOnce": True, "evaluatedAtUtc": utc_now(), "testCaseMetricRows": len(case_rows), "checkpointSha256": checkpoint_sha, "splitSha256": split_sha256()})
    atomic_write_json(OUTPUT_DIRS["metrics"] / "test_metrics.json", result)
    atomic_write_csv(OUTPUT_DIRS["metrics"] / "test_case_metrics.csv", case_rows)
    atomic_write_csv(OUTPUT_DIRS["metrics"] / "test_metrics_per_class.csv", per_class_rows(result))
    confusion_matrix_csv(OUTPUT_DIRS["metrics"] / "test_confusion_matrix.csv", result)
    save_test_predictions_preview(model, loaders["test"], OUTPUT_DIRS["figures"] / "test_predictions.png")
    TEST_EVALUATED_IN_MEMORY = True
    return result


def quality_gate(test_metrics: dict[str, Any], integrity: dict[str, Any], runtime_verification: dict[str, Any]) -> dict[str, Any]:
    reasons = []
    if not integrity.get("patientHeldout"):
        reasons.append("split_no_confirmado_por_paciente")
    if test_metrics.get("dice_macro_foreground") is None or float(test_metrics["dice_macro_foreground"]) < CFG.QUALITY_GATE_DICE_MACRO_FOREGROUND:
        reasons.append("dice_macro_foreground_below_threshold")
    if test_metrics.get("iou_macro_foreground") is None:
        reasons.append("iou_macro_foreground_missing")
    for name in ["raw0Dice", "raw0Precision", "raw0Recall"]:
        if test_metrics.get(name) is None:
            reasons.append(f"{name}_missing")
    if runtime_verification.get("shape") != [1, CFG.NUM_CLASSES, *CFG.TARGET_SIZE]:
        reasons.append("runtime_shape_invalid")
    if runtime_verification.get("finite") is not True:
        reasons.append("runtime_not_finite")
    return {
        "qualityGatePassed": not reasons,
        "thresholdDiceMacroForeground": CFG.QUALITY_GATE_DICE_MACRO_FOREGROUND,
        "diceMacroIncludingRaw0": test_metrics.get("dice_macro_foreground"),
        "diceMacroExcludingRaw0": test_metrics.get("dice_macro_excluding_raw0"),
        "iouMacroIncludingRaw0": test_metrics.get("iou_macro_foreground"),
        "iouMacroExcludingRaw0": test_metrics.get("iou_macro_excluding_raw0"),
        "runtimeVerification": runtime_verification,
        "reasons": reasons,
        "humanReviewRequired": humanReviewRequired,
        "heldOutReuseWarning": HELD_OUT_REUSE_WARNING,
        "notClinicalDiagnosis": notClinicalDiagnosis,
    }


def runtime_artifact_payload(best_checkpoint_path: Path, metrics_payload: dict[str, Any]) -> dict[str, Any]:
    checkpoint = torch.load(str(best_checkpoint_path), map_location="cpu", weights_only=False)
    return {"model_state_dict": checkpoint_state_dict(checkpoint), "num_classes": CFG.NUM_CLASSES, "base_channels": CFG.BASE_CHANNELS, "target_size": CFG.TARGET_SIZE, "classes": [CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)], "label_map": RAW_TO_CLASS_INDEX, "metrics": metrics_payload, "training_status": "pending_quality_gate", "aiServiceCommit": AI_SERVICE_COMMIT_SHA}


def generate_manifest_and_model_card(artifact_path: Path, test_metrics: dict[str, Any], validation_metrics: dict[str, Any], gate: dict[str, Any], runtime_verification: dict[str, Any], best_checkpoint_path: Path) -> dict[str, Any]:
    manifest = {
        "schemaVersion": "axial-model-manifest-v1",
        "modelKey": MODEL_KEY,
        "version": CFG.RUN_ID,
        "architecture": "AxialUNet2D",
        "artifactFile": artifact_path.name,
        "artifactSha256": sha256_stream(artifact_path),
        "checkpointSha256": sha256_stream(best_checkpoint_path),
        "dataset": "ALKAFRI/Sudirman axial T2 lumbar MRI",
        "datasetVersion": "derived_from_split_manifest",
        "splitManifestSha256": split_sha256(),
        "task": "lumbar_mri_multiclass_segmentation",
        "inputPlane": "axial",
        "classes": [CLASS_INDEX_TO_NAME[i] for i in range(CFG.NUM_CLASSES)],
        "rawToClassIndex": {str(k): v for k, v in RAW_TO_CLASS_INDEX.items()},
        "monitorMetric": CFG.AXIAL_MONITOR_METRIC,
        "raw0Boost": CFG.RAW0_WEIGHT_BOOST,
        "bestEpoch": None,
        "bestValidationMetric": validation_metrics.get(CFG.AXIAL_MONITOR_METRIC),
        "aiServiceCommit": AI_SERVICE_COMMIT_SHA,
        "metrics": {
            "test": test_metrics,
            "validation": validation_metrics,
            "precisionPerClass": {k: v.get("precision") for k, v in test_metrics.get("perClass", {}).items()},
            "recallPerClass": {k: v.get("recall") for k, v in test_metrics.get("perClass", {}).items()},
            "predictedInGtAbsentCasesPerClass": {k: v.get("predictedInGtAbsentCases") for k, v in test_metrics.get("perClass", {}).items()},
        },
        "runtimeVerification": runtime_verification,
        "trainingStatus": "validated_baseline" if gate["qualityGatePassed"] else "candidate_below_quality_gate",
        "qualityGate": gate,
        "heldOutConfirmedByPatient": True,
        "targetSize": list(CFG.TARGET_SIZE),
        "baseChannels": CFG.BASE_CHANNELS,
        "numClasses": CFG.NUM_CLASSES,
        "stateDictFormat": "checkpoint_dict_with_model_state_dict",
        "gitCommit": ENV_MANIFEST.get("gitCommit"),
        "seed": CFG.SEED,
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
        "heldOutReuseWarning": HELD_OUT_REUSE_WARNING,
    }
    atomic_write_json(manifest_path_for(artifact_path), manifest)
    model_card = f"""# Model Card - {artifact_path.name}

Configuracion v2: AxialUNet2D, base_channels={CFG.BASE_CHANNELS}, target_size={CFG.TARGET_SIZE}, raw0Boost={CFG.RAW0_WEIGHT_BOOST}.

Criterio de seleccion: {CFG.AXIAL_MONITOR_METRIC}; gate Dice macro foreground >= {CFG.QUALITY_GATE_DICE_MACRO_FOREGROUND}.

Validation metrics: {json.dumps(validation_metrics, indent=2, default=str)}

Test metrics: {json.dumps(test_metrics, indent=2, default=str)}

raw_0 precision/recall: {test_metrics.get("raw0Precision")} / {test_metrics.get("raw0Recall")}.
raw_0 predicted in GT absent cases: {test_metrics.get("raw0PredictedInGtAbsentCases")}.

Quality gate: {json.dumps(gate, indent=2, default=str)}

Runtime verification: {json.dumps(runtime_verification, indent=2, default=str)}

Limitaciones: semantica anatomica raw_* pendiente; requiere revision humana y no constituye diagnostico clinico.

{HELD_OUT_REUSE_WARNING}
"""
    atomic_write_bytes(model_card_path_for(artifact_path), model_card.encode("utf-8"))
    return manifest


def build_axial_model_from_manifest(model_manifest: dict[str, Any]) -> nn.Module:
    if model_manifest.get("modelKey") != MODEL_KEY:
        raise ValueError("modelKey incompatible")
    if model_manifest.get("architecture", "AxialUNet2D") != "AxialUNet2D":
        raise ValueError("architecture incompatible")
    return AxialUNet2D(num_classes=int(model_manifest["numClasses"]), base_channels=int(model_manifest["baseChannels"]))


def round_trip_model_from_manifest(checkpoint_path: Path, model_manifest: dict[str, Any], example: torch.Tensor) -> dict[str, Any]:
    model = build_axial_model_from_manifest(model_manifest).to(DEVICE)
    state = torch.load(str(checkpoint_path), map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint_state_dict(state), strict=True)
    model.eval()
    with torch.inference_mode():
        output = model(example.to(DEVICE))
    shape = list(output.shape)
    finite = bool(torch.isfinite(output).all().item())
    return {"shape": shape, "finite": finite}


HELD_OUT_REUSE_WARNING = "The held-out test partition was previously evaluated for the axial-full-v1\nbaseline. This v2 evaluation is comparative and should not be interpreted as a\nfully untouched external validation."


def select_artifact_path(quality_gate_passed: bool) -> Path:
    artifact_name = "axial_t2_alkafri_final_v2_best.pt" if quality_gate_passed else "axial_t2_alkafri_final_v2_candidate.pt"
    return OUTPUT_DIRS["models"] / artifact_name


def manifest_path_for(artifact_path: Path) -> Path:
    return OUTPUT_DIRS["manifests"] / f"{artifact_path.name}.manifest.json"


def model_card_path_for(artifact_path: Path) -> Path:
    return OUTPUT_DIRS["manifests"] / f"{artifact_path.name}.modelcard.md"


## Ejecucion

In [ ]:
def test_once_pipeline() -> dict[str, Any]:
    mkdirs_for_run()
    records = build_records_from_split_manifest()
    integrity = validate_split_integrity(records)
    test_metrics = evaluate_test_once(records)
    best_checkpoint_path = RESUME_DIR / "axial_t2_alkafri_v2.best_checkpoint.pt"
    validation_metrics_path = OUTPUT_DIRS["metrics"] / "validation_metrics_best.json"
    validation_metrics = json.loads(validation_metrics_path.read_text(encoding="utf-8")) if validation_metrics_path.exists() else {}
    temp_artifact = OUTPUT_DIRS["models"] / "axial_t2_alkafri_final_v2.tmp.pt"
    final_artifacts = [OUTPUT_DIRS["models"] / "axial_t2_alkafri_final_v2_best.pt", OUTPUT_DIRS["models"] / "axial_t2_alkafri_final_v2_candidate.pt"]
    for path in [temp_artifact, *final_artifacts]:
        path.unlink(missing_ok=True)
    save_checkpoint(temp_artifact, runtime_artifact_payload(best_checkpoint_path, {"test": test_metrics, "validation": validation_metrics}))
    temp_manifest = {"modelKey": MODEL_KEY, "architecture": "AxialUNet2D", "numClasses": CFG.NUM_CLASSES, "baseChannels": CFG.BASE_CHANNELS}
    example = torch.randn(1, 1, *CFG.TARGET_SIZE, dtype=torch.float32)
    runtime_verification = round_trip_model_from_manifest(temp_artifact, temp_manifest, example)
    gate = quality_gate(test_metrics, integrity, runtime_verification)
    artifact_path = select_artifact_path(gate["qualityGatePassed"])
    temp_artifact.replace(artifact_path)
    manifest = generate_manifest_and_model_card(artifact_path, test_metrics, validation_metrics, gate, runtime_verification, best_checkpoint_path)
    verification = {
        "artifactPath": artifact_path.name,
        "artifactSha256": sha256_stream(artifact_path),
        "artifactSizeBytes": artifact_path.stat().st_size,
        "manifestPath": manifest_path_for(artifact_path).name,
        "manifestSha256": sha256_stream(manifest_path_for(artifact_path)),
        "checkpointPath": best_checkpoint_path.name,
        "checkpointSha256": sha256_stream(best_checkpoint_path),
        "splitSha256": split_sha256(),
        "runtimeShape": runtime_verification["shape"],
        "runtimeFinite": runtime_verification["finite"],
        "qualityGatePassed": gate["qualityGatePassed"],
        "completedAtUtc": utc_now(),
    }
    atomic_write_json(OUTPUT_DIRS["reports"] / "final_artifact_verification.json", verification)
    marker = OUTPUT_DIRS["metrics"] / "test_evaluated_once.json"
    in_progress = OUTPUT_DIRS["metrics"] / "test_evaluation_in_progress.json"
    completed_marker = {"runId": CFG.RUN_ID, "status": "completed", "completedAtUtc": utc_now(), "checkpointSha256": sha256_stream(best_checkpoint_path), "splitSha256": split_sha256(), "confirmationTokenHash": _confirmation_hash(), "artifactPath": artifact_path.name, "manifestPath": manifest_path_for(artifact_path).name, "verificationPath": "final_artifact_verification.json"}
    atomic_write_json(marker, completed_marker)
    in_progress.unlink(missing_ok=True)
    return {"testMetrics": test_metrics, "qualityGate": gate, "artifact": artifact_path.name, "verification": verification, "manifest": manifest}


TEST_ONCE_RESULT = test_once_pipeline()
print(json.dumps(TEST_ONCE_RESULT, indent=2, default=str))


## Outputs